# Assessing and handling uncertainty when sampling extreme events

## Background and definitions

### What we mean by 'uncertainty'
There are several kinds of scientific uncertainty that arise when working with long-term projections of future climates:

1. <b>Model Uncertainty</b>, or the differences between different models (namely, how model physics, settings, or parameters can change results).
2. <b>Internal Variability</b>, or the variations inherent within the climate system itself.

### Terms used throughout this notebook
- <b>Ensemble member</b>: When a given model is run multiple times, we call the group of runs an <i>ensemble</i>. Each ensemble member represents a distinct realization featuring its own combination of model parameters. We call the the differences between ensemble members <i>internal variability</i>. 
- <b>Downscaling</b>: One ensemble member from each Cal-Adapt: Analytics Engine model was used to provide input data for the Weather Research and Forecasting (WRF) Model, allowing us to capture much smaller-scale details than the CMIP6 models. 

---

### A brief note on why extreme events benefit from special handling

Extreme events --- that is, those which occur at the tail end of a distribution --- are by definition infrequent. This infrequency leads to very small sample sizes, a fact which exacerbates model uncertainty and internal variability. 

## Story and structure

### Variable of interest
While the assessment and handling of extreme events shown here applies to any extreme event for any variable, here we choose to explore <b>precipitation</b> because it in particular tends to have very high natural background variability, for which internal variability provides an analog.

### User story
I want to examine <span style="color:red"><b>projected changes in extreme precipitation accumulations</b></span>, in particular to meet needs associated with planning and preparedness for precipitation impacts.

### Notebook structure

Part 0 -- Setup the notebook, load and process data

Part 1 -- Introduction to uncertainty:
- Part 1.1 -- Illustrate model uncertainty by showing differences across the four downscaled Cal-Adapt models
- Part 1.2 -- Visualize internal variability across the original four CMIP6 models from which the Cal-Adapt models were downscaled
- Part 1.3 -- Show model uncertainty across the available CMIP6 catalog
- Part 1.4 -- Summarize the contributions of both model uncertainty and internal variability 

Part 2 -- Handling uncertainty:
- Show our recommended strategy and go over key points

---

# Part 0: Setup

In [ ]:
%%capture
!pip install -r requirements.txt
!pip install xmip
import climakitae as ck
import xarray as xr
import pandas as pd
import numpy as np
import datetime
import intake
import dask
import matplotlib as mpl
import matplotlib.pyplot as plt
import xesmf as xe
import cartopy.crs as ccrs
import warnings
warnings.filterwarnings("ignore")
import holoviews as hv
import panel as pn
pn.extension()
hv.extension('bokeh')

In [ ]:
%%capture
from xmip.preprocessing import (
    rename_cmip6
) # will figure out how to avoid later

def cf_to_dt(ds):
    """
    convert cftime to pandas datetime -
    some GCMs have some wacky calendars
    """
    ds = ds.copy()
    if (
        type(ds.indexes['time']) not in 
        [pd.core.indexes.datetimes.DatetimeIndex]
    ): 
        datetimeindex = ds.indexes['time'].to_datetimeindex()
        ds['time'] = datetimeindex
    return ds

def calendar_align(ds):
    '''
    different models have different calendars
    (some no leap, some 360 day). this results
    in a huge dataset with lots of empty
    values in time when concatenated. 
    the following function sets the day for all monthly
    values to the 1st of each month -
    WARNING this can impact functions which use
    the number of days in each month (eg
    precip flux to total monthly accumulation).
    I can't find a nice way around this.
    '''
    ds['time'] = pd.to_datetime(ds.time.dt.strftime('%Y-%m'))
    return ds

def precip_flux_to_total(ds):
    """
    converts precip flux units 
    (kg m-2 s-1) to total precip
    per month (mm)
    NOTE: assumes regular calendar
    """
    ds = ds.copy()
    ds_attrs = ds.attrs
    days_month = ds.time.dt.days_in_month
    seconds_month = 86400*days_month
    ds = ds*seconds_month
    ds.attrs = ds_attrs
    ds.pr.attrs["units"] = 'mm'
    return ds


def wrapper(ds):
    ds_simulation = ds.attrs["source_id"]
    ds_scenario = ds.attrs["experiment_id"]
    ds_freq = ds.attrs["frequency"]
    
    ds = ds.copy()
    ds = rename_cmip6(ds)
    # ds = broadcast_lonlat(ds)
    # ds = replace_x_y_nominal_lat_lon(ds)
    ds = cf_to_dt(ds)
    if ds_freq in ('mon'):
        ds = calendar_align(ds)
    ds = precip_flux_to_total(ds)
    ds = ds.drop_vars(["lon","lat"],
                     errors="ignore")
    ds = ds.assign_coords({'simulation' : 
                           ds_simulation,
                          'scenario' :
                          ds_scenario})
    ds = ds.squeeze(drop=True)
    
    return ds

In [ ]:
shp_cat = intake.open_catalog(
    "https://cadcat.s3.amazonaws.com/parquet/catalog.yaml")

us_states = shp_cat.states.read()
us_counties = shp_cat.counties.read()
us_watersheds = shp_cat.huc8.read()

shp_cat = None

col = intake.open_esm_datastore("https://cadcat.s3.amazonaws.com/tmp/cmip6-regrid.json")


In [ ]:
class cmip_opt():
    def __init__(self, variable='tas', 
                  area_subset='states', 
                 location='California',
                 timescale='monthly',
                area_average=True):
        self.variable = variable
        self.area_subset = area_subset
        self.location = location
        self.area_average = area_average
        self.timescale = timescale
        
    def cmip_clip(self, ds):
        ds = ds.copy()
        variable = self.variable
        location = self.location
        area_average = self.area_average
        area_subset = self.area_subset
        timescale = self.timescale
        
        to_drop = [v for v in list(
                    ds.data_vars) 
                  if v != variable]
        ds = ds.drop_vars(to_drop)
        ds = clip_region(ds,area_subset,location)
        if area_average:
            ds = area_wgt_average(ds)
        return ds
    
def clip_region(ds,area_subset,location):
    """
    clips CMIP6 dataset using a polygon.
    ds is the dataset
    state is a string ("California")
    (check catalog for other names)
    opt = 'True' to burn in all cells
    which touch the boundary (keep as False)
    """
    ds = ds.copy()
    ds = ds.rio.write_crs(4326)
    
    if 'counties' in area_subset:
        ds_region = us_counties[us_counties.NAME 
                    == location].geometry
    elif 'states' in area_subset:
        ds_region = us_states[us_states.NAME 
                    == location].geometry
        
    try:
        ds = ds.rio.clip(
            geometries=ds_region,crs=4326, drop=True,
        all_touched=False)
    except: 
        #if no grid centers in region
        # instead select all cells which 
        # intersect the region
        print('USER NOTE: Selecting all cells which intersect region')
        ds = ds.rio.clip(
            geometries=ds_region,crs=4326, drop=True,
        all_touched=True)
        
    return ds

def area_wgt_average(ds):
    ds = ds.copy()
    weights = np.cos(np.deg2rad(ds.y))
    weights.name = "weights"
    ds_weighted = ds.weighted(weights)
    ds = ds_weighted.mean(("x","y"))
    return ds

def drop_member_id(dset_dict): 
    """Drop member_id coordinate/dimensions 
    Args: 
        dset_dict (dict): dictionary in the format {dataset_name:xr.Dataset}
    """
    for dname, dset in dset_dict.items():
        if "member_id" in dset.coords: 
            dset = dset.isel(member_id=0).drop("member_id") # Drop coord
            dset_dict.update({dname: dset}) # Update dataset in dictionary
    return dset_dict

In [ ]:
from climakitae.utils import _read_ae_colormap

def _make_hvplot_wrf(data, title, vmin, vmax, sopt, width=300, height=300):
    """Make single map"""
    if sopt:
        cmap = _read_ae_colormap(cmap='ae_diverging', cmap_hex=True) # sets to a diverging colormap
    else:
        cmap = _read_ae_colormap(cmap='ae_blue', cmap_hex=True)     
    _plot = data.hvplot.quadmesh(
        x="lon",
        y="lat",
        projection=ccrs.PlateCarree(),
        grid=True,
        width=width,
        height=height,
        xaxis=None,
        yaxis=None,
        symmetric=sopt,
        vmin=vmin,
        vmax=vmax,
        clim=(vmin,vmax),
        title=title,
        clabel="Precipitation (mm/month)",
        features=["coastline"],
        cmap=cmap,
        )
    
    # _plot.opts(xlim=xlim, ylim=ylim)

    
    return _plot

In [ ]:
#Set options for CMIP6 data
copt = cmip_opt()
copt.variable = 'pr'
copt.area_subset = 'states' 
copt.location = 'California'
copt.area_average = False
copt.timescale = 'monthly'

sim_names = {
    "CESM2": "cesm2",
    "CNRM-ESM2-1": "cnrm-esm2-1",
    "EC-Earth3-Veg": "ec-earth3-veg",
    "FGOALS-g3": "fgoals-g3"
}

cmip_names = list(sim_names.keys())

cae_colors_list = [
    '#4477AA', '#66CCEE',
    '#228833', '#EE6677',
    '#AA3377' ,
]

In [ ]:
app = ck.Application()
app.selections.scenario_historical=['Historical Climate']
app.selections.scenario_ssp=['SSP 3-7.0 -- Business as Usual']
app.selections.append_historical = True
app.selections.area_average = 'No'
app.selections.variable = 'Precipitation (total)'
app.selections.time_slice = (1981, 2100)
app.selections.resolution = '9 km'
app.location.area_subset='states'
app.location.cached_area = 'CA'

wrf_ds = app.retrieve()

In [ ]:
renamed_ds_list = []

for cn,wn in sim_names.items():
    renamed_ds_list.append(
        wrf_ds.sel(simulation=wn
                  ).squeeze().assign_coords(
            {'simulation' : cn}))
    
wrf_ds = xr.concat(renamed_ds_list,dim='simulation')
hist_wrf = wrf_ds.sel(
    time=slice('1981','2010'))
ssp_wrf = wrf_ds.sel(
    time=slice('2071','2100'))

# mean and 99th percentile all models

cads_hist_percentile = hist_wrf.chunk(dict(
    time=-1)).quantile([.99],
    dim='time').compute().squeeze()

cads_ssp_percentile = ssp_wrf.chunk(dict(
    time=-1)).quantile([.99],
    dim='time').compute().squeeze()

cads_delta_percentile = (cads_ssp_percentile 
                         - cads_hist_percentile).compute()

# 1: Introduction to uncertainty

## 1.1: Model uncertainty

We begin by illustrating differences in results across the four downscaled Cal-Adapt models.

In [ ]:
hv.extension('bokeh')
vmin = 0
vmax = 2200
num_simulations = 4

cads_hist_plots = _make_hvplot_wrf(
        data=cads_hist_percentile.isel(simulation=0),
        title=(cads_hist_percentile.isel(simulation=0
                                       ).simulation.item()
              ),
        vmin=vmin,
        vmax=vmax,
        sopt=False)

for sim_i in range(1, num_simulations): # plot remaining simulations
    pl_i = _make_hvplot_wrf(
        data=cads_hist_percentile.isel(simulation=sim_i),
        title=(cads_hist_percentile.isel(simulation=sim_i
                                       ).simulation.item()
              ),
        vmin=vmin,
        vmax=vmax,
        sopt=False)
    cads_hist_plots += pl_i

# additional aesthetic settings to tidy figure
cads_hist_plots.cols(4)  # organize columns
cads_hist_plots.opts(hv.opts.Layout(merge_tools=True))  # merge toolbar
cads_hist_plots.opts(toolbar="below")  # set toolbar location
cads_hist_plots.opts(
            title="99th percentile monthly precipitation accumulation"
            + " across models, 1981-2010"
        )  # add title

# now for EOC
cads_ssp_plots = _make_hvplot_wrf(
        data=cads_ssp_percentile.isel(simulation=0),
        title=(cads_ssp_percentile.isel(simulation=0
                                      ).simulation.item()
              ),
        vmin=vmin,
        vmax=vmax,
        sopt=False)

for sim_i in range(1, num_simulations): # plot remaining simulations
    pl_i = _make_hvplot_wrf(
        data=cads_ssp_percentile.isel(simulation=sim_i),
        title=(cads_ssp_percentile.isel(simulation=sim_i
                                      ).simulation.item()
              ),
        vmin=vmin,
        vmax=vmax,
        sopt=False)
    cads_ssp_plots += pl_i

# additional aesthetic settings to tidy figure
cads_ssp_plots.cols(4)  # organize columns
cads_ssp_plots.opts(hv.opts.Layout(merge_tools=True))  # merge toolbar
cads_ssp_plots.opts(toolbar="below")  # set toolbar location
cads_ssp_plots.opts(
            title="99th percentile monthly precipitation accumulation"
            + " across models, 2071-2100"
        )  # add title

# now for the difference
cads_diff_plots = _make_hvplot_wrf(
        data=cads_delta_percentile.isel(simulation=0),
        title=(cads_delta_percentile.isel(simulation=0
                                      ).simulation.item()
              ),
        vmin=-600,
        vmax=600,
        sopt=True)

for sim_i in range(1, num_simulations): # plot remaining simulations
    pl_i = _make_hvplot_wrf(
        data=cads_delta_percentile.isel(simulation=sim_i),
        title=(cads_delta_percentile.isel(simulation=sim_i
                                      ).simulation.item()
              ),
        vmin=-600,
        vmax=600,
        sopt=True)
    cads_diff_plots += pl_i

# additional aesthetic settings to tidy figure
cads_diff_plots.cols(4)  # organize columns
cads_diff_plots.opts(hv.opts.Layout(merge_tools=True))  # merge toolbar
cads_diff_plots.opts(toolbar="below")  # set toolbar location
cads_diff_plots.opts(
            title="Change in 99th percentile monthly precipitation accumulation"
            + "  across models by end-of-century"
        )  # add title

cads_tabs = pn.Tabs(('Present-day',cads_hist_plots), 
               ('End-of-century',cads_ssp_plots),
               ('Difference',cads_diff_plots), 
               dynamic=False)
cads_tabs

#### Interpretation

Looking at all Cal-Adapt models reveals quite a bit of disagreement in precipitation accumulations, as well as precipitation sign and magnitude changes by end-of-century. These plots show that sampling from a single simulation can limit our understanding, and comparing across simulations can give conflicting and confusing results which are difficult to interpret from a practical perspective. In the next section, we will elucidate a significant contributor to these differences.

## 1.2: Internal variability

Internal variability describes the intra-model differences, that is the differences which occur across a given model's ensemble (recall that an ensemble is a series of model runs which feature small tweaks in initial conditions, and sometimes use different underlying physics which govern smaller scale processes). Internal variability results from the inherently chaotic nature of the climate system, and as a result provides an analog for natural background variability.  

<b>We illustrate internal variability</b> by comparing the ensemble members of the CMIP6 models which provided the large-scale input for the downscaled Cal-Adapt models shown above. Hereon we call these models <i>parent models</i>.

In [ ]:
cat = col.search(
    table_id = "Amon",
    variable_id = 'pr',
    experiment_id = ["historical","ssp370"],
    source_id = cmip_names,
)
dsets = cat.to_dataset_dict(zarr_kwargs={'consolidated': True}, 
                            storage_options={'anon': True},
                           preprocess=wrapper
                           )
# sort by models so indexing is easy
dsets = dict(sorted(dsets.items()))

# Subsets the historical scenario
hist_dsets = {key: val for key,val in dsets.items()
             if "historical" in key}

# Subsets the future scenario
ssp_dsets = {key: val for key,val in dsets.items()
               if "ssp370" in key}

In [ ]:
hist_list = list(hist_dsets.values())
ssp_list = list(ssp_dsets.values())
num_simulations = len(sim_names)
sim_range = range(num_simulations)


hist_cae_ds = [copt.cmip_clip(ds) for
                ds in hist_list]
ssp_cae_ds = [copt.cmip_clip(ds) for
                ds in ssp_list]
# pick ensemble member subset from ssp simulations
common_members = [list(ssp_cae_ds[s
                    ].member_id.values)
                  for s in sim_range]
hist_cae_ds = [hist_cae_ds[s].sel(
                member_id=common_members[s]
                ) for s in sim_range]
sorted_sim_names = [hist_cae_ds[s].attrs['parent_source_id']
                    for s in sim_range]

# percentile for the cal-adapt GCMs (together)
hist_cae_ds = [hist_cae_ds[s].sel(
                time=slice('1981','2010')
                ) for s in sim_range]
ssp_cae_ds = [ssp_cae_ds[s].sel(
                time=slice('2071','2100')
                ) for s in sim_range]

hist_cae_percentile = [hist_cae_ds[s].chunk(
    dict(time=-1)).quantile([.99],
    dim='time').compute() for s in sim_range]

ssp_cae_percentile = [ssp_cae_ds[s].chunk(
    dict(time=-1)).quantile([.99],
    dim='time').compute() for s in sim_range]

cae_delta_percentile = [(ssp_cae_percentile[s] 
                        - hist_cae_percentile[s]
                       ).compute() for s in sim_range]


In [ ]:
all_ens_plots = []

dat_list = [hist_cae_percentile,
             ssp_cae_percentile,
             cae_delta_percentile]
sopt_list = [False, False, True]
vlim_list = [[0,700],[0,700],[-150,150]]

for i in range(3):
    dat_coll = dat_list[i]
    sopt = sopt_list[i]
    vlim = vlim_list[i]
    all_ens = []
    for sim_i in range(num_simulations):
        num_members = len(dat_coll[sim_i].member_id)

        dat = dat_coll[sim_i].rename(
                            {'x' : 'lon',
                           'y' : 'lat'}
        )
        _plots = _make_hvplot_wrf(
                data=dat.isel(member_id=0).squeeze(),
                title=(sorted_sim_names[sim_i]
                      + " member " + str(1)),
                vmin=vlim[0],
                vmax=vlim[1],
                sopt=sopt)

        for m_i in range(1, num_members): # plot remaining simulations
            pl_i = _make_hvplot_wrf(
                data=dat.isel(member_id=m_i).squeeze(),
                title=(sorted_sim_names[sim_i]
                      + " member " + str(m_i+1)),
                vmin=vlim[0],
                vmax=vlim[1],
                sopt=sopt)
            _plots += pl_i

        # additional aesthetic settings to tidy figure
        _plots.cols(4)  # organize columns
        _plots.opts(hv.opts.Layout(merge_tools=True))  # merge toolbar
        _plots.opts(toolbar="below")  # set toolbar location
        all_ens.append(_plots)   
    all_ens_plots.append(all_ens)
    
all_hist_ens = (all_ens_plots[0][0]
                + all_ens_plots[0][1]
                + all_ens_plots[0][2]
                + all_ens_plots[0][3])
all_hist_ens.opts(
            title="99th percentile monthly precipitation accumulation"
            + " across ensemble members, 1981-2010"
        )  

all_ssp_ens = (all_ens_plots[1][0]
                + all_ens_plots[1][1]
                + all_ens_plots[1][2]
                + all_ens_plots[1][3])
all_ssp_ens.opts(
            title="99th percentile monthly precipitation accumulation"
            + " across ensemble members, 2071-2100"
        )  

all_diff_ens = (all_ens_plots[2][0]
                + all_ens_plots[2][1]
                + all_ens_plots[2][2]
                + all_ens_plots[2][3])
all_diff_ens.opts(
            title="Change in 99th percentile monthly precipitation accumulation"
            + " across ensemble members by end-of-century"
        )  

pn.Tabs(('Present-day',all_hist_ens),
        ('End-of-century',all_ssp_ens),
        ('Difference',all_diff_ens))

#### Interpretation
We have now expanded our analysis to the larger model ensembles from which the Cal-Adapt models were derived. Doing so shows us that not only are differences across models significant, but so are differences within models. Since the downscaled Cal-Adapt models come from one ensemble member, <b>the model differences shown in the first set of maps come from a combination of model uncertainty and internal variability</b>. 

## 1.3: Model uncertainty across the CMIP6 catalog

In [ ]:
# grab data from the cmip6 archive
cat = col.search(
    table_id = "Amon",
    variable_id = copt.variable,
    experiment_id = ["historical","ssp370"],
    member_id = "r1i1p1f1",
    require_all_on="source_id"
).search(activity_id = ["CMIP","ScenarioMIP"])

dsets = cat.to_dataset_dict(
    zarr_kwargs={'consolidated': True}, 
    storage_options={'anon': True},
    preprocess=wrapper)
dsets = {key: val for key,val in dsets.items() 
         if 'BCC-ESM1' not in key}
dsets = {key: val for key,val in dsets.items() 
         if 'MPI-ESM-1-2-HAM' not in key}
dsets = dict(sorted(dsets.items()))


# subsets the historical scenario
hist_dsets = {key: val for key,val in dsets.items()
             if "historical" in key}

# subsets the future scenario
ssp_dsets = {key: val for key,val in dsets.items()
               if "ssp370" in key}

# grab the additional cal-adapt simulations
paths = ['CESM2.*r11i1p1f1', 'CNRM-ESM2-1.*r1i1p1f2', 'MPI-ESM1-2-LR.*r7i1p1f1']
cat = col.search(table_id="Amon", 
           variable_id=copt.variable, 
           path=paths, 
           activity_id=['CMIP', 'ScenarioMIP'])

cal_dsets = cat.to_dataset_dict(
    zarr_kwargs={'consolidated': True}, 
    storage_options={'anon': True},
    preprocess=wrapper)
cal_dsets = dict(sorted(cal_dsets.items()))


# subsets the historical scenario
cal_hist_dsets = {key: val for key,val in cal_dsets.items()
             if "historical" in key}

# subsets the future scenario
cal_ssp_dsets = {key: val for key,val in cal_dsets.items()
               if "ssp370" in key}

In [ ]:
# drop member id 
hist_dsets = drop_member_id(hist_dsets)
cal_hist_dsets = drop_member_id(cal_hist_dsets)
ssp_dsets = drop_member_id(ssp_dsets) 
cal_ssp_dsets = drop_member_id(cal_ssp_dsets)


# merge dataset together
all_hist_mdls = hist_dsets | cal_hist_dsets
all_ssp_mdls = ssp_dsets | cal_ssp_dsets

hist_ds = xr.concat(list(all_hist_mdls.values()), dim='simulation').squeeze()
hist_ds = copt.cmip_clip(hist_ds.sel(time=slice('1850','2014')))

ssp_ds = xr.concat(list(all_ssp_mdls.values()), dim='simulation').squeeze()
ssp_ds = copt.cmip_clip(ssp_ds)

In [ ]:
# 99th percentile all models

hist_percentile = hist_ds.sel(
    time=slice('1981','2010')).chunk(dict(
    time=-1)).quantile([.99],
    dim='time').compute().squeeze()

ssp_percentile = ssp_ds.sel(
    time=slice('2071','2100')).chunk(dict(
    time=-1)).quantile([.99],
    dim='time').compute().squeeze()

delta_percentile = (ssp_percentile 
                    - hist_percentile).compute()



In [ ]:
hv.extension('bokeh')
vmin = 0
vmax = 700
num_simulations = len(hist_percentile.simulation)

hist_plots = _make_hvplot_wrf(
        data=hist_percentile.isel(simulation=0
                                 ).rename({'x' : 'lon',
                                'y' : 'lat'}),
        title=(hist_percentile.isel(simulation=0
                                       ).simulation.item()
              ),
        vmin=vmin,
        vmax=vmax,
        sopt=False)

for sim_i in range(1, num_simulations): # plot remaining simulations
    pl_i = _make_hvplot_wrf(
        data=hist_percentile.isel(simulation=sim_i
                                 ).rename({'x' : 'lon',
                                'y' : 'lat'}),
        title=(hist_percentile.isel(simulation=sim_i
                                       ).simulation.item()
              ),
        vmin=vmin,
        vmax=vmax,
        sopt=False)
    hist_plots += pl_i

# additional aesthetic settings to tidy figure
hist_plots.cols(4)  # organize columns
hist_plots.opts(hv.opts.Layout(merge_tools=True))  # merge toolbar
hist_plots.opts(toolbar="below")  # set toolbar location
hist_plots.opts(
            title="99th percentile monthly precipitation accumulation"
            + " in each model, 1981-2010"
        )  # add title

# now for EOC
ssp_plots = _make_hvplot_wrf(
        data=ssp_percentile.isel(simulation=0
                                 ).rename({'x' : 'lon',
                                'y' : 'lat'}),
        title=(ssp_percentile.isel(simulation=0
                                      ).simulation.item()
              ),
        vmin=vmin,
        vmax=vmax,
        sopt=False)

for sim_i in range(1, num_simulations): # plot remaining simulations
    pl_i = _make_hvplot_wrf(
        data=ssp_percentile.isel(simulation=sim_i
                                 ).rename({'x' : 'lon',
                                'y' : 'lat'}),
        title=(ssp_percentile.isel(simulation=sim_i
                                      ).simulation.item()
              ),
        vmin=vmin,
        vmax=vmax,
        sopt=False)
    ssp_plots += pl_i

# additional aesthetic settings to tidy figure
ssp_plots.cols(4)  # organize columns
ssp_plots.opts(hv.opts.Layout(merge_tools=True))  # merge toolbar
ssp_plots.opts(toolbar="below")  # set toolbar location
ssp_plots.opts(
            title="99th percentile monthly precipitation accumulation"
            + " in each model, 2071-2100"
        )  # add title

# now for the difference
diff_plots = _make_hvplot_wrf(
        data=delta_percentile.isel(simulation=0
                                 ).rename({'x' : 'lon',
                                'y' : 'lat'}),
        title=(delta_percentile.isel(simulation=0
                                      ).simulation.item()
              ),
        vmin=-150,
        vmax=150,
        sopt=True)

for sim_i in range(1, num_simulations): # plot remaining simulations
    pl_i = _make_hvplot_wrf(
        data=delta_percentile.isel(simulation=sim_i
                                 ).rename({'x' : 'lon',
                                'y' : 'lat'}),
        title=(delta_percentile.isel(simulation=sim_i
                                      ).simulation.item()
              ),
        vmin=-150,
        vmax=150,
        sopt=True)
    diff_plots += pl_i

# additional aesthetic settings to tidy figure
diff_plots.cols(4)  # organize columns
diff_plots.opts(hv.opts.Layout(merge_tools=True))  # merge toolbar
diff_plots.opts(toolbar="below")  # set toolbar location
diff_plots.opts(
            title="Change in 99th percentile monthly precipitation accumulation"
            + " in each model by end-of-century"
        )  # add title

tabs = pn.Tabs(('1981-2010',hist_plots), 
               ('2071-2100',ssp_plots),
               ('Difference',diff_plots), 
               dynamic=False)
tabs

### Interpretation 
[in-progress] Differences across models shown above occur both as a result of actual model differences and internal variability. In the next section, we will show how these two sources of uncertainty compare to one another. 

## 1.4: Uncertainty summarized

We will compactly summarize model uncertainty and internal variability by examining the area-average change over a small region (e.g., Sonoma County). Note that <b>here we analyze relative (percent) changes rather than absolute changes</b>, since the magnitude of extreme precipitation in the downscaled Cal-Adapt models is much larger than in the non-downscaled runs (see the first set of maps versus the subsequent figures). 

In [ ]:
app.selections.area_average = 'Yes'
app.selections.time_slice = (1981, 2100)
app.selections.resolution = '9 km'
app.location.area_subset='CA counties'
app.location.cached_area = 'Sonoma County'
reg_wrf_ds = app.retrieve()

renamed_ds_list = []

for cn,wn in sim_names.items():
    renamed_ds_list.append(
        reg_wrf_ds.sel(simulation=wn
                  ).squeeze().assign_coords(
            {'simulation' : cn}))

In [ ]:
reg_wrf_ds = xr.concat(renamed_ds_list,dim='simulation')
reg_hist_wrf = reg_wrf_ds.sel(
    time=slice('1981','2010'))
reg_ssp_wrf = reg_wrf_ds.sel(
    time=slice('2071','2100'))

reg_cads_hist_percentile = reg_hist_wrf.chunk(dict(
    time=-1)).quantile([.99],
    dim='time').compute().squeeze()

reg_cads_ssp_percentile = reg_ssp_wrf.chunk(dict(
    time=-1)).quantile([.99],
    dim='time').compute().squeeze()

In [ ]:
reg_cads_delta_percentile = (((reg_cads_ssp_percentile 
                         - reg_cads_hist_percentile)
                             /reg_cads_hist_percentile
                             )*100).compute()

cae_rel_delta_percentile = [((cae_delta_percentile[s] 
                            / hist_cae_percentile[s]
                           )*100).compute() for s in sim_range]

reg_cae_delta_percentile = [area_wgt_average(clip_region(
                            cae_rel_delta_percentile[s],'counties',
                            app.location.cached_area)).squeeze() 
                            for s in sim_range]

bar_heights = [reg_cae_delta_percentile[s].pr.max(
            dim="member_id",skipna=True).item() 
            for s in sim_range]
bar_floors = [reg_cae_delta_percentile[s].pr.min(
            dim="member_id",skipna=True).item() 
            for s in sim_range]
line_pos = [reg_cae_delta_percentile[s].pr.median(
            dim="member_id",skipna=True).item() 
            for s in sim_range]

fig,ax = plt.subplots(figsize=(8,4))
ax.bar(x=reg_cads_delta_percentile.simulation.values,
       height=bar_heights,
       bottom=bar_floors,width=.2,ec='k',
      alpha=.5,label="Ensemble range")
ax.scatter(reg_cads_delta_percentile['simulation'],
           line_pos,zorder=2,ec='k',
          s=1000,marker="_",label="Ensemble median")
ax.scatter(reg_cads_delta_percentile['simulation'],
           reg_cads_delta_percentile,zorder=2,ec='k',fc='y',
          s=100,label="Downscaled member")
ax.legend()
ax.set_title("Relative change in 99th percentile monthly precipitation accumulation, "+app.location.cached_area)
ax.set_ylabel("% change")
ax.set_xlabel("Model name")
plt.show()

### Interpretation
Both internal variability (shown by the blue bars) and differences across models are large, making it difficult to use the data presented above to form actionable conclusions. The next section shows a reasonable analysis technique in light of the uncertainty we have illustrated so far. 

# 2: Handling uncertainty

## Recommended strategy: Pool the data
Given that internal variability is high compared to model uncertainty (which we show above), and assuming that a model run appropriately captures the relevant processes for extreme precipitation, we can treat each ensemble member of each model as an equally plausible representation of the climate. <b>Under this treatment, we can pool the data from all simulations (which increases our sampling by a factor of four) and sample from the larger distribution.</b>

In [ ]:
hist_wrf_pool = hist_wrf.stack(index=['simulation','time'])
ssp_wrf_pool = ssp_wrf.stack(index=['simulation','time'])
hist_wrf_pool_perc = hist_wrf_pool.chunk(
    dict(index=-1)).quantile([.99],
    dim='index').compute().squeeze()

ssp_wrf_pool_perc = ssp_wrf_pool.chunk(
    dict(index=-1)).quantile([.99],
    dim='index').compute().squeeze()

delta_wrf_pool_perc = (ssp_wrf_pool_perc
                       - hist_wrf_pool_perc
                      ).compute()

mmm_hist_plot = _make_hvplot_wrf(
        data=hist_wrf_pool_perc,
        title=("Pooled"),
        vmin=0,
        vmax=2000,
        sopt=False)
mmm_hist_plot.opts(toolbar="below")  # set toolbar location

mmm_ssp_plot = _make_hvplot_wrf(
        data=ssp_wrf_pool_perc,
        title=("Pooled"),
        vmin=0,
        vmax=2000,
        sopt=False)
mmm_ssp_plot.opts(toolbar="below")  # set toolbar location

mmm_diff_plot = _make_hvplot_wrf(
        data=delta_wrf_pool_perc,
        title=("Pooled"),
        vmin=-300,
        vmax=300,
        sopt=True)
mmm_diff_plot.opts(toolbar="below")  # set toolbar location

pooled_hist =mmm_hist_plot + cads_hist_plots
pooled_ssp = mmm_ssp_plot + cads_ssp_plots
pooled_diff = mmm_diff_plot + cads_diff_plots

pool_cads_tabs = pn.Tabs(('Present-day',pooled_hist), 
               ('End-of-century',pooled_ssp),
               ('Difference',pooled_diff), 
               dynamic=False)

### Show results after pooling with the per-model results from Section 1.1 for reference

In [ ]:
pool_cads_tabs

# Summary and key points

As discussed in Section 1.2, the model uncertainty shown in this notebook occurs as a function of both actual model differences and internal variability.
* The uncertainty related to these sources is exacerbated by small sample size.
* The 'Difference' results above show that cross-model results can be in conflict with one another, with some models showing marked decreases in extreme precipitation where others show large increases. This makes it difficult to use these projections for impacts research and planning and preparedness purposes.

Here we overcame some of these issues by pooling the data across models before computing statistics, a strategy which provided the following advantages:
* Increased the underlying sample size from which to derive statistics.
* Provided interpretable results which are useful from an impacts perspective.